# 02_block_cam_sensitivity_mel_nv

Qualitative block sensitivity check for MEL vs NV CAMs.

Goal:
- Compare where heatmaps focus across transformer blocks.
- Use this as an internal visual analysis notebook.
- Notebook 03 can then do quantitative block analysis.

Default blocks:
`[-1, -2, -4, -6, -8, -10, -12]`

Output design:
- Generate panels for CLS and GAP separately.
- Build one PDF per model.
- Each PDF page = one image.
- Each page has one row per block.
- Columns: RGB lesion outline, Grad CAM target, Diff CAM, Finer CAM.


In [1]:
from pathlib import Path
import json
import subprocess
import shlex
import pandas as pd

# =============================================================================
# 1. MAIN PARAMETERS
# =============================================================================

# Notebook location: notebooks/mel_nv/02_block_cam_sensitivity_mel_nv.ipynb
# REPO_ROOT = Path("../..").resolve() # local notebook
REPO_ROOT = Path("..").resolve() # ubelix notebook
REPO_ROOT = REPO_ROOT / "master-thesis"
HAM_ROOT = REPO_ROOT / "data" / "HAM10000"
MEL_NV_ROOT = HAM_ROOT / "mel_nv"

IMG_DIR = HAM_ROOT
MASK_ROOT = HAM_ROOT

# Keep qualitative first. all_test with 7 blocks x 2 models is heavy.
# Recommended: qualitative
SAMPLE_MODE = "qualitative"  # "qualitative" or "all_test"

# Requested qualitative block sweep
TARGET_BLOCK_INDICES = [-1, -2, -4, -6, -8, -10, -12]

# Set True first if you only want to inspect commands.
DRY_RUN = False

# If None, use all rows in the selected CSV.
# For faster debugging, set e.g. NUM_SAMPLES_OVERRIDE = 2
NUM_SAMPLES_OVERRIDE = None

# For block sensitivity, keep columns compact.
# Full version possible: rgb_gt_mask,gradcam_a,gradcam_b,map_diff,finercam
PANEL_ITEMS = "rgb_gt_mask,gradcam_a,map_diff,finercam"
PANEL_SUFFIX = PANEL_ITEMS.replace(",", "_")

# Finer-CAM comparison strength
ALPHA = 0.8

# Quantitative CAM lesion alignment
TOP_HEAT_RATIO = 0.10
CAM_METRIC_METHOD = "finercam"  # "gradcam_target", "gradcam_diff", or "finercam"

# Paths
CLEAN_CSV = MEL_NV_ROOT / "ham_mel_nv_clean.csv"
QUAL_CSV = MEL_NV_ROOT / "ham_mel_nv_clean_qualitative_10_per_class_seed42.csv"

CLS_CKPT = REPO_ROOT / "external" / "checkpoints4" / "checkpoint-best-cls.pth"
GAP_CKPT = REPO_ROOT / "external" / "checkpoints4" / "checkpoint-best-gap.pth"

GT_COL = "gt_label"
CLASS_ARGS = ["--class_names", "MEL,NV"]
COMPARE_ARGS = [
    "--compare_mode", "gt_pair",
    "--A", "MEL",
    "--B", "NV",
    "--topk_compare", "1",
]

SCENARIOS = [
    {
        "name": "CLS HA 0.5",
        "short_name": "cls_ha05",
        "checkpoint": CLS_CKPT,
        "checkpoint_model_type": "panderm",
        "pooling": "cls",
    },
    {
        "name": "GAP HA 0.25",
        "short_name": "gap_ha025",
        "checkpoint": GAP_CKPT,
        "checkpoint_model_type": "panderm",
        "pooling": "mean",
    },
]

OUT_ROOT = REPO_ROOT / "outputs" / "mel_nv" / f"block_cam_sensitivity_{SAMPLE_MODE}"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("OUT_ROOT:", OUT_ROOT)
print("CLEAN_CSV exists:", CLEAN_CSV.exists(), CLEAN_CSV)
print("QUAL_CSV exists:", QUAL_CSV.exists(), QUAL_CSV)
print("CLS_CKPT exists:", CLS_CKPT.exists(), CLS_CKPT)
print("GAP_CKPT exists:", GAP_CKPT.exists(), GAP_CKPT)
print("Blocks:", TARGET_BLOCK_INDICES)

for scenario in SCENARIOS:
    print("\n", scenario["name"])
    print("  checkpoint:", scenario["checkpoint"])
    print("  pooling:", scenario["pooling"])
    if not Path(scenario["checkpoint"]).exists():
        print("  [WARN] missing checkpoint")


REPO_ROOT: /storage/homefs/cn21m021/projects/master-thesis
OUT_ROOT: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_qualitative
CLEAN_CSV exists: True /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000/mel_nv/ham_mel_nv_clean.csv
QUAL_CSV exists: True /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000/mel_nv/ham_mel_nv_clean_qualitative_10_per_class_seed42.csv
CLS_CKPT exists: True /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints4/checkpoint-best-cls.pth
GAP_CKPT exists: True /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints4/checkpoint-best-gap.pth
Blocks: [-1, -2, -4, -6, -8, -10, -12]

 CLS HA 0.5
  checkpoint: /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints4/checkpoint-best-cls.pth
  pooling: cls

 GAP HA 0.25
  checkpoint: /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints4/checkpoint-best-gap.pth
  pooling: mean


## 2. Build active CSV


In [2]:
def safe_name(name: str) -> str:
    out = str(name).lower()
    for ch in [" ", "/", "\\", ":", ";", ",", "(", ")", "[", "]", "{", "}", "+", "."]:
        out = out.replace(ch, "_")
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")


def make_active_csv() -> tuple[Path, int, pd.DataFrame]:
    csv_out_dir = OUT_ROOT / "csv"
    csv_out_dir.mkdir(parents=True, exist_ok=True)

    if SAMPLE_MODE == "qualitative":
        active_csv = QUAL_CSV
        df = pd.read_csv(active_csv)
    elif SAMPLE_MODE == "all_test":
        df = pd.read_csv(CLEAN_CSV)
        df = df[df["split"].astype(str).str.lower().eq("test")].copy()
        df = df.sort_values(["gt_label", "image_id"]).reset_index(drop=True)
        active_csv = csv_out_dir / "ham_mel_nv_clean_all_test.csv"
        df.to_csv(active_csv, index=False)
    else:
        raise ValueError("SAMPLE_MODE must be 'qualitative' or 'all_test'.")

    if NUM_SAMPLES_OVERRIDE is None:
        num_samples = len(df)
    else:
        num_samples = min(int(NUM_SAMPLES_OVERRIDE), len(df))

    print("ACTIVE_CSV:", active_csv)
    print("NUM_SAMPLES:", num_samples)
    display(df.head())
    display(df.head(num_samples).groupby(["split", "gt_label"]).size().unstack(fill_value=0))
    return active_csv, num_samples, df.head(num_samples).copy()


ACTIVE_CSV, NUM_SAMPLES, DISPLAY_DF = make_active_csv()


ACTIVE_CSV: /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000/mel_nv/ham_mel_nv_clean_qualitative_10_per_class_seed42.csv
NUM_SAMPLES: 20


,lesion_id,image_id,image,dx,gt_label,label,label_2class,binary_label,split,image_rel_path,...,cue_applied,cue_mask_rel_path,cue_mode,dx_type,age,sex,localization,dataset,age_group,dx_norm
0,HAM_0005846,ISIC_0024459,ISIC_0024459.jpg,mel,MEL,4,0,0,test,images/ISIC_0024459.jpg,...,False,NaN,clean,histo,80.0,male,back,vienna_dias,old,mel
1,HAM_0007272,ISIC_0024756,ISIC_0024756.jpg,mel,MEL,4,0,0,test,images/ISIC_0024756.jpg,...,False,NaN,clean,histo,60.0,male,lower extremity,rosendahl,old,mel
2,HAM_0002576,ISIC_0025414,ISIC_0025414.jpg,mel,MEL,4,0,0,test,images/ISIC_0025414.jpg,...,False,NaN,clean,histo,55.0,male,lower extremity,rosendahl,old,mel
3,HAM_0007031,ISIC_0025616,ISIC_0025616.jpg,mel,MEL,4,0,0,test,images/ISIC_0025616.jpg,...,False,NaN,clean,histo,70.0,female,upper extremity,rosendahl,old,mel
4,HAM_0006696,ISIC_0026094,ISIC_0026094.jpg,mel,MEL,4,0,0,test,images/ISIC_0026094.jpg,...,False,NaN,clean,histo,20.0,male,back,rosendahl,young,mel


gt_label,MEL,NV
split,,
test,10,10


## 3. Generate CAM panels for all blocks

This calls `scripts.generate_finer_cam_panderm` once per model and block.

Output folder pattern:

`outputs/mel_nv/block_cam_sensitivity_<mode>/panels/<model>/block_<block>/`


In [ ]:
def run_command(cmd: list[str], dry_run: bool = False):
    print("\n" + "=" * 100)
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 100)
    if dry_run:
        return
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def block_dir_name(block_index: int) -> str:
    return f"block_{block_index}".replace("-", "minus")


def generate_panels():
    panel_root = OUT_ROOT / "panels"
    panel_root.mkdir(parents=True, exist_ok=True)

    for scenario in SCENARIOS:
        for block_index in TARGET_BLOCK_INDICES:
            scenario_out_dir = panel_root / scenario["short_name"] / block_dir_name(block_index)
            scenario_out_dir.mkdir(parents=True, exist_ok=True)

            cmd = [
                "python", "-m", "scripts.generate_finer_cam_panderm",
                "--csv", str(ACTIVE_CSV),
                "--image_col", "image_rel_path",
                "--img_dir", str(IMG_DIR),
                "--gt_col", GT_COL,
                "--checkpoint", str(scenario["checkpoint"]),
                "--checkpoint_model_type", scenario.get("checkpoint_model_type", "panderm"),
                "--pooling", scenario["pooling"],
                "--out_dir", str(scenario_out_dir),
                "--num_samples", str(NUM_SAMPLES),
                "--method", "finercam",
                "--alpha", str(ALPHA),
                "--panel_items", PANEL_ITEMS,
                "--mask_root", str(MASK_ROOT),
                "--mask_col", "mask_rel_path",
                "--target_block_index", str(block_index),
                "--clinician_labels",
                "--model_display_name", f"{scenario['name']} block {block_index}",
                # "--save_json",
                "--save_raw_cams",
            ]
            cmd += CLASS_ARGS
            cmd += COMPARE_ARGS

            print(f"\nGenerating: {scenario['name']} | block {block_index}")
            run_command(cmd, dry_run=DRY_RUN)


generate_panels()


## 4. Quantitative lesion alignment per block

This computes simple lesion alignment metrics from saved raw CAMs:

- `top10_inside`: fraction of top 10% hottest CAM pixels inside the lesion mask
- `pointing_game`: whether the hottest CAM pixel is inside the lesion mask
- `inside_mean`: mean CAM value inside lesion
- `outside_mean`: mean CAM value outside lesion

These metrics do not prove clinical correctness, but they help choose a model/block where the heatmap is at least lesion-oriented.

In [4]:
import numpy as np
from PIL import Image
import cv2

def image_id_to_stem(image_id_value: str) -> str:
    p = Path(str(image_id_value))
    stem = p.stem if p.suffix else p.name
    return stem.replace("/", "_").replace("\\", "_").replace(" ", "_")


def resolve_from_root(root: Path, value) -> Path:
    p = Path(str(value))
    if p.is_absolute():
        return p
    return (root / p).resolve()


def minmax_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x).squeeze().astype(np.float32)
    x = x - np.nanmin(x)
    denom = np.nanmax(x) + 1e-8
    return x / denom


def load_mask_for_row(row: pd.Series, target_hw: tuple[int, int]) -> np.ndarray:
    if "mask_rel_path" not in row.index or pd.isna(row["mask_rel_path"]):
        raise ValueError("mask_rel_path missing")

    mask_path = resolve_from_root(MASK_ROOT, row["mask_rel_path"])
    if not mask_path.exists():
        raise FileNotFoundError(mask_path)

    mask = Image.open(mask_path).convert("L")
    mask = np.array(mask)
    mask = (mask > 0).astype(np.uint8)

    h, w = target_hw
    if mask.shape[:2] != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

    return mask.astype(bool)


def image_id_to_stem_from_row(row: pd.Series) -> str:
    if "image_id" in row.index and pd.notna(row["image_id"]):
        return image_id_to_stem(row["image_id"])
    if "image_rel_path" in row.index and pd.notna(row["image_rel_path"]):
        return image_id_to_stem(row["image_rel_path"])
    if "image" in row.index and pd.notna(row["image"]):
        return image_id_to_stem(row["image"])
    raise ValueError("Could not infer image id/stem from row.")


def find_raw_cam_file(scenario: dict, block_index: int, row: pd.Series, method: str) -> Path | None:
    out_dir = OUT_ROOT / "panels" / scenario["short_name"] / block_dir_name(block_index)
    raw_dir = out_dir / "raw_cams"

    if not raw_dir.exists():
        return None

    stem_candidates = []
    if "image_id" in row.index and pd.notna(row["image_id"]):
        stem_candidates.append(image_id_to_stem(row["image_id"]))
    if "image_rel_path" in row.index and pd.notna(row["image_rel_path"]):
        stem_candidates.append(image_id_to_stem(row["image_rel_path"]))
    if "image" in row.index and pd.notna(row["image"]):
        stem_candidates.append(image_id_to_stem(row["image"]))

    # Common likely patterns
    for stem in dict.fromkeys(stem_candidates):
        patterns = [
            f"{stem}_{method}.npy",
            f"{stem}_cam_{method}.npy",
            f"{stem}*{method}*.npy",
        ]
        for pattern in patterns:
            matches = sorted(raw_dir.glob(pattern))
            if matches:
                return matches[0]

    # Fallback: inspect all npy files for this image stem
    for stem in dict.fromkeys(stem_candidates):
        matches = sorted(raw_dir.glob(f"{stem}*.npy"))
        method_matches = [p for p in matches if method in p.name]
        if method_matches:
            return method_matches[0]
        if matches and method == "finercam":
            # fallback only if names are not method-specific
            return matches[0]

    return None


def compute_alignment_metrics(cam: np.ndarray, mask: np.ndarray, top_ratio: float = 0.10) -> dict:
    cam = minmax_np(cam)

    if cam.shape[:2] != mask.shape[:2]:
        h, w = mask.shape[:2]
        cam = cv2.resize(cam, (w, h), interpolation=cv2.INTER_LINEAR)
        cam = minmax_np(cam)

    mask_bool = mask.astype(bool)

    if mask_bool.sum() == 0:
        return {
            "top10_inside": np.nan,
            "pointing_game": np.nan,
            "inside_mean": np.nan,
            "outside_mean": np.nan,
        }

    flat_cam = cam.reshape(-1)
    flat_mask = mask_bool.reshape(-1)

    k = max(1, int(round(float(top_ratio) * flat_cam.size)))
    top_idx = np.argpartition(flat_cam, -k)[-k:]
    top_inside = float(flat_mask[top_idx].mean())

    max_idx = int(np.argmax(flat_cam))
    pointing_game = float(flat_mask[max_idx])

    inside_mean = float(cam[mask_bool].mean())
    outside_mean = float(cam[~mask_bool].mean()) if (~mask_bool).sum() > 0 else np.nan

    return {
        "top10_inside": top_inside,
        "pointing_game": pointing_game,
        "inside_mean": inside_mean,
        "outside_mean": outside_mean,
    }


def compute_block_alignment_table():
    rows = []

    for scenario in SCENARIOS:
        for block_index in TARGET_BLOCK_INDICES:
            for _, row in DISPLAY_DF.iterrows():
                image_id = str(row.get("image_id", image_id_to_stem_from_row(row)))
                gt_label = str(row.get(GT_COL, row.get("gt_label", "unknown")))

                cam_path = find_raw_cam_file(
                    scenario=scenario,
                    block_index=block_index,
                    row=row,
                    method=CAM_METRIC_METHOD,
                )

                if cam_path is None:
                    rows.append({
                        "model": scenario["name"],
                        "model_short": scenario["short_name"],
                        "gt_label": gt_label,
                        "image_id": image_id,
                        "block": block_index,
                        "cam_method": CAM_METRIC_METHOD,
                        "top10_inside": np.nan,
                        "pointing_game": np.nan,
                        "inside_mean": np.nan,
                        "outside_mean": np.nan,
                        "status": "missing_cam",
                    })
                    continue

                try:
                    cam = np.load(cam_path)
                    cam = minmax_np(cam)
                    mask = load_mask_for_row(row, target_hw=cam.shape[:2])
                    metrics = compute_alignment_metrics(cam, mask, top_ratio=TOP_HEAT_RATIO)

                    rows.append({
                        "model": scenario["name"],
                        "model_short": scenario["short_name"],
                        "gt_label": gt_label,
                        "image_id": image_id,
                        "block": block_index,
                        "cam_method": CAM_METRIC_METHOD,
                        **metrics,
                        "status": "ok",
                        "cam_path": str(cam_path),
                    })

                except Exception as e:
                    rows.append({
                        "model": scenario["name"],
                        "model_short": scenario["short_name"],
                        "gt_label": gt_label,
                        "image_id": image_id,
                        "block": block_index,
                        "cam_method": CAM_METRIC_METHOD,
                        "top10_inside": np.nan,
                        "pointing_game": np.nan,
                        "inside_mean": np.nan,
                        "outside_mean": np.nan,
                        "status": "failed",
                        "error": str(e),
                        "cam_path": str(cam_path),
                    })

    return pd.DataFrame(rows)


alignment_df = compute_block_alignment_table()

alignment_out = OUT_ROOT / f"block_cam_alignment_{CAM_METRIC_METHOD}_top{int(TOP_HEAT_RATIO * 100)}.csv"
alignment_df.to_csv(alignment_out, index=False)

print("Saved per-image alignment table:", alignment_out)
display(alignment_df[[
    "model",
    "gt_label",
    "image_id",
    "block",
    "top10_inside",
    "pointing_game",
    "inside_mean",
    "outside_mean",
    "status",
]].head(20))

print("Status counts:")
display(alignment_df["status"].value_counts(dropna=False))

Saved per-image alignment table: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_qualitative/block_cam_alignment_finercam_top10.csv


,model,gt_label,image_id,block,top10_inside,pointing_game,inside_mean,outside_mean,status
0,CLS HA 0.5,MEL,ISIC_0024459,-1,0.842766,1.0,0.223351,0.090903,ok
1,CLS HA 0.5,MEL,ISIC_0024756,-1,0.651654,1.0,0.416225,0.164402,ok
2,CLS HA 0.5,MEL,ISIC_0025414,-1,0.989438,1.0,0.223289,0.053547,ok
3,CLS HA 0.5,MEL,ISIC_0025616,-1,0.523316,0.0,0.332461,0.249816,ok
4,CLS HA 0.5,MEL,ISIC_0026094,-1,0.057393,0.0,0.136675,0.217515,ok
5,CLS HA 0.5,MEL,ISIC_0026993,-1,0.691909,0.0,0.454890,0.127290,ok
6,CLS HA 0.5,MEL,ISIC_0028897,-1,0.640096,1.0,0.317700,0.109530,ok
7,CLS HA 0.5,MEL,ISIC_0030360,-1,0.158629,1.0,0.306504,0.250092,ok
8,CLS HA 0.5,MEL,ISIC_0030798,-1,0.377441,0.0,0.122743,0.158348,ok
9,CLS HA 0.5,MEL,ISIC_0031408,-1,0.664807,0.0,0.138651,0.095900,ok


Status counts:


status
ok    280
Name: count, dtype: int64

## 5. Build block sensitivity PDFs

This creates one PDF per model:
- `block_cam_sensitivity_cls_ha05_<mode>.pdf`
- `block_cam_sensitivity_gap_ha025_<mode>.pdf`

Each page is one image. Each row is one transformer block.

In [5]:
from PIL import Image, ImageDraw, ImageFont


def get_font(size: int, bold: bool = False):
    candidates = [
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for path in candidates:
        if path and Path(path).exists():
            return ImageFont.truetype(path, size=size)
    return ImageFont.load_default()


FONT_TITLE = get_font(34, bold=True)
FONT_SUBTITLE = get_font(22, bold=False)
FONT_LABEL = get_font(22, bold=True)
FONT_SMALL = get_font(17, bold=False)


def find_panel_png(scenario: dict, block_index: int, row: pd.Series) -> Path | None:
    out_dir = OUT_ROOT / "panels" / scenario["short_name"] / block_dir_name(block_index)
    candidates = []
    if "image_rel_path" in row and pd.notna(row["image_rel_path"]):
        candidates.append(image_id_to_stem(row["image_rel_path"]))
    if "image_id" in row and pd.notna(row["image_id"]):
        candidates.append(image_id_to_stem(row["image_id"]))
    if "image" in row and pd.notna(row["image"]):
        candidates.append(image_id_to_stem(row["image"]))

    for stem in dict.fromkeys(candidates):
        direct = out_dir / f"{stem}_{PANEL_SUFFIX}.png"
        if direct.exists():
            return direct
        matches = sorted(out_dir.glob(f"{stem}_*.png"))
        if matches:
            return matches[0]
    return None


def make_page_for_image_and_model(row: pd.Series, scenario: dict) -> Image.Image:
    page_width = 2200
    margin = 45
    label_width = 210
    gap = 14
    title_h = 115
    available_panel_width = page_width - 2 * margin - label_width - gap

    loaded_panels = []
    for block_index in TARGET_BLOCK_INDICES:
        panel_path = find_panel_png(scenario, block_index, row)
        if panel_path is None:
            loaded_panels.append((block_index, None, None))
            continue

        panel = Image.open(panel_path).convert("RGB")
        scale = available_panel_width / panel.width
        new_h = int(panel.height * scale)
        panel = panel.resize((available_panel_width, new_h), Image.Resampling.LANCZOS)
        loaded_panels.append((block_index, panel, panel_path))

    row_heights = [panel.height if panel is not None else 190 for _, panel, _ in loaded_panels]
    page_height = title_h + margin + sum(row_heights) + gap * (len(row_heights) - 1) + margin
    page = Image.new("RGB", (page_width, page_height), "white")
    draw = ImageDraw.Draw(page)

    image_id = row.get("image_id", row.get("image_rel_path", "unknown"))
    gt = row.get(GT_COL, row.get("gt_label", "unknown"))
    title = f"Block CAM sensitivity: {scenario['name']}"
    subtitle = f"Image: {image_id} | Ground truth: {gt} | Pooling: {scenario['pooling']} | Mode: {SAMPLE_MODE}"
    draw.text((margin, 26), title, fill="black", font=FONT_TITLE)
    draw.text((margin, 75), subtitle, fill=(60, 60, 60), font=FONT_SUBTITLE)

    y = title_h
    for block_index, panel, panel_path in loaded_panels:
        row_h = panel.height if panel is not None else 190
        label_x = margin
        label_y = y + 20
        draw.text((label_x, label_y), f"Block {block_index}", fill="black", font=FONT_LABEL)
        draw.text((label_x, label_y + 32), "qualitative", fill=(80, 80, 80), font=FONT_SMALL)

        if panel is None:
            box_x = margin + label_width + gap
            box_y = y
            draw.rectangle([box_x, box_y, box_x + available_panel_width, box_y + row_h], outline=(180, 180, 180), width=2)
            draw.text((box_x + 30, box_y + 65), "Missing panel PNG", fill=(160, 0, 0), font=FONT_LABEL)
        else:
            page.paste(panel, (margin + label_width + gap, y))
        y += row_h + gap

    return page


def build_pdf_for_scenario(scenario: dict):
    pages = []
    for _, row in DISPLAY_DF.iterrows():
        pages.append(make_page_for_image_and_model(row, scenario))

    if not pages:
        raise RuntimeError("No pages generated.")

    pdf_out = OUT_ROOT / f"block_cam_sensitivity_{scenario['short_name']}_{SAMPLE_MODE}.pdf"
    pages[0].save(pdf_out, save_all=True, append_images=pages[1:], resolution=150.0)
    print("Saved PDF:", pdf_out)
    return pdf_out


PDF_OUTPUTS = []
if DRY_RUN:
    print("DRY_RUN=True, skipping PDF build.")
else:
    for scenario in SCENARIOS:
        PDF_OUTPUTS.append(build_pdf_for_scenario(scenario))


Saved PDF: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_qualitative/block_cam_sensitivity_cls_ha05_qualitative.pdf
Saved PDF: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_qualitative/block_cam_sensitivity_gap_ha025_qualitative.pdf


## 6. Save config and quick checks


In [6]:
config = {
    "sample_mode": SAMPLE_MODE,
    "target_block_indices": TARGET_BLOCK_INDICES,
    "num_samples": NUM_SAMPLES,
    "active_csv": str(ACTIVE_CSV),
    "out_root": str(OUT_ROOT),
    "panel_items": PANEL_ITEMS,
    "alpha": ALPHA,
    "pdf_outputs": [str(p) for p in PDF_OUTPUTS],
    "scenarios": [
        {
            "name": s["name"],
            "short_name": s["short_name"],
            "checkpoint": str(s["checkpoint"]),
            "checkpoint_model_type": s.get("checkpoint_model_type", "panderm"),
            "pooling": s["pooling"],
        }
        for s in SCENARIOS
    ],
    "class_args": CLASS_ARGS,
    "compare_args": COMPARE_ARGS,
}

config_out = OUT_ROOT / f"block_cam_sensitivity_config_{SAMPLE_MODE}.json"
config_out.write_text(json.dumps(config, indent=2))
print("Saved config:", config_out)

for scenario in SCENARIOS:
    print("\n" + "=" * 80)
    print(scenario["name"])
    for block_index in TARGET_BLOCK_INDICES:
        out_dir = OUT_ROOT / "panels" / scenario["short_name"] / block_dir_name(block_index)
        pngs = sorted(out_dir.glob("*.png"))
        metas = sorted(out_dir.glob("*_meta.json"))
        raw_dir = out_dir / "raw_cams"
        raw_files = sorted(raw_dir.glob("*.npy")) if raw_dir.exists() else []
        print(f"  block {block_index:>3}: png={len(pngs):>4} meta={len(metas):>4} raw_flat={len(raw_files):>4}")

print("\nPDF outputs:")
for p in PDF_OUTPUTS:
    print(" ", p)


Saved config: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_qualitative/block_cam_sensitivity_config_qualitative.json

CLS HA 0.5
  block  -1: png=  20 meta=   0 raw_flat=  80
  block  -2: png=  20 meta=   0 raw_flat=  80
  block  -4: png=  20 meta=   0 raw_flat=  80
  block  -6: png=  20 meta=   0 raw_flat=  80
  block  -8: png=  20 meta=   0 raw_flat=  80
  block -10: png=  20 meta=   0 raw_flat=  80
  block -12: png=  20 meta=   0 raw_flat=  80

GAP HA 0.25
  block  -1: png=  20 meta=   0 raw_flat=  80
  block  -2: png=  20 meta=   0 raw_flat=  80
  block  -4: png=  20 meta=   0 raw_flat=  80
  block  -6: png=  20 meta=   0 raw_flat=  80
  block  -8: png=  20 meta=   0 raw_flat=  80
  block -10: png=  20 meta=   0 raw_flat=  80
  block -12: png=  20 meta=   0 raw_flat=  80

PDF outputs:
  /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_qualitative/block_cam_sensitivity_cls_ha05_qualitative.pdf
  /storage/hom

## 7. Practical interpretation checklist

When reviewing the PDFs, look for:

- Does the heatmap stay inside the lesion outline?
- Does it focus on lesion border, pigment network, dark/irregular areas, or clinically plausible structures?
- Does it focus on artifacts such as hair, dark corners, ruler marks, or background?
- Do later blocks become more concentrated or more diffuse?
- Does CLS behave differently from GAP?
- Which block gives stable maps across both MEL and NV examples?

Do not choose the final block only from this notebook. Use this to form hypotheses, then confirm quantitatively in notebook 03.

In [7]:
ok_alignment_df = alignment_df[alignment_df["status"].eq("ok")].copy()

if len(ok_alignment_df) == 0:
    raise RuntimeError("No valid CAM alignment rows found. Check raw CAM file naming/path.")

summary_cols = [
    "top10_inside",
    "pointing_game",
    "inside_mean",
    "outside_mean",
]

alignment_summary = (
    ok_alignment_df
    .groupby(["model", "model_short", "gt_label", "block"], dropna=False)
    .agg(
        n=("image_id", "count"),
        top10_inside=("top10_inside", "mean"),
        pointing_game=("pointing_game", "mean"),
        inside_mean=("inside_mean", "mean"),
        outside_mean=("outside_mean", "mean"),
    )
    .reset_index()
)

alignment_summary["inside_outside_gap"] = (
    alignment_summary["inside_mean"] - alignment_summary["outside_mean"]
)

summary_out = OUT_ROOT / f"block_cam_alignment_summary_by_class_{CAM_METRIC_METHOD}_top{int(TOP_HEAT_RATIO * 100)}.csv"
alignment_summary.to_csv(summary_out, index=False)

print("Saved summary by class:", summary_out)
display(alignment_summary)


overall_summary = (
    ok_alignment_df
    .groupby(["model", "model_short", "block"], dropna=False)
    .agg(
        n=("image_id", "count"),
        top10_inside=("top10_inside", "mean"),
        pointing_game=("pointing_game", "mean"),
        inside_mean=("inside_mean", "mean"),
        outside_mean=("outside_mean", "mean"),
    )
    .reset_index()
)

overall_summary["inside_outside_gap"] = (
    overall_summary["inside_mean"] - overall_summary["outside_mean"]
)

# Simple selection score:
# - top10_inside is most important
# - pointing_game is second
# - inside_outside_gap helps break ties
overall_summary["selection_score"] = (
    0.50 * overall_summary["top10_inside"]
    + 0.35 * overall_summary["pointing_game"]
    + 0.15 * overall_summary["inside_outside_gap"]
)

overall_out = OUT_ROOT / f"block_cam_alignment_summary_overall_{CAM_METRIC_METHOD}_top{int(TOP_HEAT_RATIO * 100)}.csv"
overall_summary.to_csv(overall_out, index=False)

print("Saved overall summary:", overall_out)
display(overall_summary.sort_values(["model", "selection_score"], ascending=[True, False]))


print("\n" + "=" * 100)
print("FINAL NUMERIC BLOCK RECOMMENDATION")
print("=" * 100)

for model_name, sub in overall_summary.groupby("model"):
    sub_sorted = sub.sort_values(
        ["selection_score", "top10_inside", "pointing_game", "inside_outside_gap"],
        ascending=False,
    ).reset_index(drop=True)

    best = sub_sorted.iloc[0]

    print(
        f"{model_name}: choose block {int(best['block'])} "
        f"based on numeric lesion alignment "
        f"(score={best['selection_score']:.3f}, "
        f"top10_inside={best['top10_inside']:.3f}, "
        f"pointing_game={best['pointing_game']:.3f}, "
        f"inside_mean={best['inside_mean']:.3f}, "
        f"outside_mean={best['outside_mean']:.3f})."
    )

print("=" * 100)
print("Important: use this as numeric support, then confirm visually in the PDF before final clinician material.")
print("=" * 100)

Saved summary by class: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_qualitative/block_cam_alignment_summary_by_class_finercam_top10.csv


,model,model_short,gt_label,block,n,top10_inside,pointing_game,inside_mean,outside_mean,inside_outside_gap
0,CLS HA 0.5,cls_ha05,MEL,-12,10,0.243902,0.2,0.259373,0.356034,-0.096661
1,CLS HA 0.5,cls_ha05,MEL,-10,10,0.194241,0.1,0.055326,0.148961,-0.093634
2,CLS HA 0.5,cls_ha05,MEL,-8,10,0.203129,0.1,0.066823,0.158709,-0.091886
3,CLS HA 0.5,cls_ha05,MEL,-6,10,0.294619,0.4,0.076104,0.137766,-0.061663
4,CLS HA 0.5,cls_ha05,MEL,-4,10,0.672599,0.6,0.344319,0.124525,0.219794
5,CLS HA 0.5,cls_ha05,MEL,-2,10,0.304942,0.2,0.157541,0.208553,-0.051012
6,CLS HA 0.5,cls_ha05,MEL,-1,10,0.559745,0.5,0.267249,0.151734,0.115515
7,CLS HA 0.5,cls_ha05,NV,-12,10,0.455959,0.4,0.220995,0.061749,0.159246
8,CLS HA 0.5,cls_ha05,NV,-10,10,0.597190,0.5,0.326528,0.129187,0.197341
9,CLS HA 0.5,cls_ha05,NV,-8,10,0.403049,0.3,0.253719,0.178850,0.074869


Saved overall summary: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/block_cam_sensitivity_qualitative/block_cam_alignment_summary_overall_finercam_top10.csv


,model,model_short,block,n,top10_inside,pointing_game,inside_mean,outside_mean,inside_outside_gap,selection_score
6,CLS HA 0.5,cls_ha05,-1,20,0.404853,0.35,0.200557,0.194571,0.005986,0.325824
1,CLS HA 0.5,cls_ha05,-10,20,0.395715,0.30,0.190927,0.139074,0.051853,0.310636
4,CLS HA 0.5,cls_ha05,-4,20,0.372250,0.30,0.193419,0.199835,-0.006415,0.290163
0,CLS HA 0.5,cls_ha05,-12,20,0.349930,0.30,0.240184,0.208891,0.031293,0.284659
3,CLS HA 0.5,cls_ha05,-6,20,0.255460,0.35,0.122320,0.185410,-0.063090,0.240767
2,CLS HA 0.5,cls_ha05,-8,20,0.303089,0.20,0.160271,0.168780,-0.008509,0.220268
5,CLS HA 0.5,cls_ha05,-2,20,0.265753,0.25,0.148465,0.198516,-0.050051,0.212869
13,GAP HA 0.25,gap_ha025,-1,20,0.414926,0.45,0.222312,0.231618,-0.009306,0.363567
8,GAP HA 0.25,gap_ha025,-10,20,0.423446,0.40,0.201705,0.133670,0.068036,0.361928
12,GAP HA 0.25,gap_ha025,-2,20,0.398535,0.35,0.225709,0.202674,0.023036,0.325223



FINAL NUMERIC BLOCK RECOMMENDATION
CLS HA 0.5: choose block -1 based on numeric lesion alignment (score=0.326, top10_inside=0.405, pointing_game=0.350, inside_mean=0.201, outside_mean=0.195).
GAP HA 0.25: choose block -1 based on numeric lesion alignment (score=0.364, top10_inside=0.415, pointing_game=0.450, inside_mean=0.222, outside_mean=0.232).
Important: use this as numeric support, then confirm visually in the PDF before final clinician material.
